# Benchmark: Deep Learning Baselines

**Models** : Vanilla LSTM · CNN-LSTM  
**Author** : Anwesha Singh — CSE, Manipal University Jaipur

---

These baselines share the **same Bidirectional DRNN architecture** as the  
proposed NiOA-optimised model, but use **fixed, manually chosen hyperparameters**  
rather than NiOA-tuned ones.  This isolates the contribution of the  
optimisation algorithm from the architecture itself.

Splits are loaded from the canonical frozen arrays.


In [ ]:
import os, sys, json, gc, warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import tensorflow.keras.backend as K
from tensorflow.keras.models    import Sequential
from tensorflow.keras.layers    import (
    Input, Conv1D, MaxPooling1D,
    LSTM, Bidirectional, Dropout,
    BatchNormalization, GlobalMaxPooling1D, Dense
)
from tensorflow.keras.callbacks import EarlyStopping

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".." , ".."))
sys.path.insert(0, PROJECT_ROOT)

from core.config  import SPLITS_DIR, BENCHMARK_DIR, RANDOM_SEED
from core.utils   import configure_gpu, set_seeds, make_tf_dataset
from benchmarking.utils.data_loader import load_splits
from benchmarking.utils.metrics     import save_benchmark_results, build_comparison_table

configure_gpu()
set_seeds(RANDOM_SEED)
sns.set(style="whitegrid")
print("Environment ready.")

In [ ]:
# ============================================================
HORIZON = 60
# ============================================================

X_train, y_train, X_val, y_val, X_test, y_test, scaler, meta = \
    load_splits(SPLITS_DIR, HORIZON)

SEQ_LEN   = X_train.shape[1]
NUM_FEATS = X_train.shape[2]
BATCH     = 32
EPOCHS    = 40
PATIENCE  = 7

train_ds = make_tf_dataset(X_train, y_train, SEQ_LEN, NUM_FEATS, BATCH)
val_ds   = make_tf_dataset(X_val,   y_val,   SEQ_LEN, NUM_FEATS, BATCH)

print(f"Horizon k = {HORIZON} s | seq={SEQ_LEN} | feats={NUM_FEATS}")

---
## 1 · Vanilla LSTM (fixed hyperparameters)

In [ ]:
K.clear_session(); gc.collect()

lstm_model = Sequential(name="Vanilla_LSTM")
lstm_model.add(Input(shape=(SEQ_LEN, NUM_FEATS)))
lstm_model.add(LSTM(64, return_sequences=True))
lstm_model.add(Dropout(0.3))
lstm_model.add(LSTM(64, return_sequences=False))
lstm_model.add(Dropout(0.3))
lstm_model.add(Dense(32, activation="relu"))
lstm_model.add(Dense(1))
lstm_model.compile(loss="mse", optimizer="adam", metrics=["mae"])

cb = [EarlyStopping(monitor="val_loss", patience=PATIENCE,
                    restore_best_weights=True, verbose=1)]

history_lstm = lstm_model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS,
    steps_per_epoch  = len(X_train) // BATCH,
    validation_steps = len(X_val)   // BATCH,
    callbacks=cb, verbose=1
)

y_pred_lstm = lstm_model.predict(X_test, verbose=0).flatten()
metrics_lstm = save_benchmark_results(
    "vanilla_lstm", HORIZON, y_test, y_pred_lstm, BENCHMARK_DIR
)
print("Vanilla LSTM →", metrics_lstm)

---
## 2 · CNN-LSTM

In [ ]:
K.clear_session(); gc.collect()

cnn_lstm = Sequential(name="CNN_LSTM")
cnn_lstm.add(Input(shape=(SEQ_LEN, NUM_FEATS)))
cnn_lstm.add(Conv1D(filters=64, kernel_size=3, activation="relu", padding="same"))
cnn_lstm.add(MaxPooling1D(pool_size=2))
cnn_lstm.add(Conv1D(filters=32, kernel_size=3, activation="relu", padding="same"))
cnn_lstm.add(LSTM(64, return_sequences=False))
cnn_lstm.add(Dropout(0.3))
cnn_lstm.add(Dense(32, activation="relu"))
cnn_lstm.add(Dense(1))
cnn_lstm.compile(loss="mse", optimizer="adam", metrics=["mae"])

history_cnn = cnn_lstm.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS,
    steps_per_epoch  = len(X_train) // BATCH,
    validation_steps = len(X_val)   // BATCH,
    callbacks=cb, verbose=1
)

y_pred_cnn = cnn_lstm.predict(X_test, verbose=0).flatten()
metrics_cnn = save_benchmark_results(
    "cnn_lstm", HORIZON, y_test, y_pred_cnn, BENCHMARK_DIR
)
print("CNN-LSTM →", metrics_cnn)

---
## 3 · Results Summary

In [ ]:
table = build_comparison_table(BENCHMARK_DIR, HORIZON)
print(f"All saved benchmarks — k = {HORIZON} s")
print(table.to_string(index=False))
out_csv = os.path.join(BENCHMARK_DIR, f"horizon_{HORIZON}", "summary_all.csv")
table.to_csv(out_csv, index=False)
print(f"\nSaved to : {out_csv}")